# Download STEAD

Downloads the STEAD dataset in SeisBench format, so that `seisbench.data.STEAD()` loads it directly from the cache.

Runs unchanged on **Google Colab** and **locally** (macOS / Linux / Windows).

| File | Size |
| --- | --- |
| `metadata.csv` | ~0.4 GB |
| `waveforms.hdf5` | ~91 GB |

- Files go to `<data dir>/seisbench/datasets/stead/`. The data dir is `F3S_DATA_DIR` if set, otherwise `./data` in the repo (on Colab: Google Drive or `/content/data`, see settings).
- The download is **resumable**: if the session or the connection drops, run the notebook again and it continues where it stopped.
- To only explore the metadata first, set `DOWNLOAD_WAVEFORMS = False`.

**Colab note:** 91 GB does not fit in the free Google Drive quota (15 GB), and a standard Colab runtime may not have enough local disk either. The disk-space cell below checks this before downloading.

## 1. Settings

In [ ]:
DOWNLOAD_WAVEFORMS = True   # False: download metadata.csv only (~0.4 GB)

# Colab only
USE_DRIVE = True            # True: save to Google Drive (persists between sessions); False: /content/data (lost on disconnect)
DRIVE_DATA_SUBDIR = "first-three-seconds/data"   # relative to MyDrive

# Mirrors, tried in order. Both serve the same files.
MIRRORS = [
    "https://hifis-storage.desy.de/Helmholtz/HelmholtzAI/SeisBench/datasets/stead/",
    "https://seisbench.gfz.de/mirror/datasets/stead/",
]

## 2. Environment

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
print(f"Python {sys.version.split()[0]} | Colab: {IN_COLAB}")

if importlib.util.find_spec("seisbench") is None:
    if IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "seisbench"], check=True)
    else:
        raise ImportError("seisbench is not installed. Run: pip install -r requirements.txt (or pip install seisbench)")

if "seisbench" in sys.modules:
    print("WARNING: seisbench was already imported in this kernel; its cache path is fixed. Restart the kernel if the path below is wrong.")

## 3. Data directory

Priority: `F3S_DATA_DIR` environment variable → Colab (Drive or `/content/data`) → `<repo>/data`.

`SEISBENCH_CACHE_ROOT` must be set **before** `seisbench` is imported, so that `seisbench.data.STEAD()` finds the files.

In [ ]:
def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / ".git").exists() or (p / "pyproject.toml").is_file() or (p / "spec.md").is_file():
            return p
    return start


if os.environ.get("F3S_DATA_DIR"):
    data_dir = Path(os.environ["F3S_DATA_DIR"]).expanduser()
elif IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        data_dir = Path("/content/drive/MyDrive") / DRIVE_DATA_SUBDIR
    else:
        data_dir = Path("/content/data")
else:
    data_dir = find_repo_root(Path.cwd().resolve()) / "data"

data_dir = data_dir.resolve()
os.environ["F3S_DATA_DIR"] = str(data_dir)
os.environ["SEISBENCH_CACHE_ROOT"] = str(data_dir / "seisbench")

stead_dir = data_dir / "seisbench" / "datasets" / "stead"
stead_dir.mkdir(parents=True, exist_ok=True)
print("Data dir: ", data_dir)
print("STEAD dir:", stead_dir)

## 4. Resumable downloader

In [ ]:
import time

import requests
from tqdm.auto import tqdm

CHUNK_BYTES = 8 * 1024 * 1024
MAX_RETRIES = 20


def remote_size(url: str) -> int:
    r = requests.head(url, allow_redirects=True, timeout=30)
    r.raise_for_status()
    return int(r.headers["Content-Length"])


def pick_mirror(file_name: str) -> tuple[str, int]:
    for base in MIRRORS:
        url = base + file_name
        try:
            return url, remote_size(url)
        except requests.RequestException as e:
            print(f"Mirror failed: {url} ({e})")
    raise RuntimeError(f"No mirror available for {file_name}")


def download(file_name: str) -> Path:
    # Writes to <name>.download and renames when complete. Never uses SeisBench's own '.partial' name.
    target = stead_dir / file_name
    tmp = stead_dir / (file_name + ".download")
    url, total = pick_mirror(file_name)

    if target.is_file():
        if target.stat().st_size == total:
            print(f"{file_name}: already complete ({total / 1e9:.2f} GB)")
            return target
        print(f"{file_name}: size mismatch, resuming via temporary file")
        target.rename(tmp)

    for attempt in range(1, MAX_RETRIES + 1):
        done = tmp.stat().st_size if tmp.is_file() else 0
        if done > total:
            tmp.unlink()
            done = 0
        if done == total:
            break
        headers = {"Range": f"bytes={done}-"} if done else {}
        try:
            with requests.get(url, headers=headers, stream=True, timeout=60) as r:
                r.raise_for_status()
                if done and r.status_code != 206:
                    print("Server ignored the Range header, restarting from 0")
                    done = 0
                mode = "ab" if done else "wb"
                with open(tmp, mode) as f, tqdm(total=total, initial=done, unit="B", unit_scale=True,
                                                unit_divisor=1024, desc=file_name) as bar:
                    for chunk in r.iter_content(chunk_size=CHUNK_BYTES):
                        f.write(chunk)
                        bar.update(len(chunk))
        except (requests.RequestException, OSError) as e:
            wait = min(60, 2 ** attempt)
            print(f"Attempt {attempt}/{MAX_RETRIES} interrupted: {e}. Retrying in {wait} s")
            time.sleep(wait)
    else:
        raise RuntimeError(f"{file_name}: download did not finish after {MAX_RETRIES} attempts; run the cell again")

    size = tmp.stat().st_size
    if size != total:
        raise RuntimeError(f"{file_name}: expected {total} bytes, got {size}; run the cell again")
    tmp.replace(target)
    print(f"{file_name}: done ({total / 1e9:.2f} GB)")
    return target

## 5. Disk space check

In [ ]:
import shutil

files = ["metadata.csv"] + (["waveforms.hdf5"] if DOWNLOAD_WAVEFORMS else [])

needed = 0
for name in files:
    _, total = pick_mirror(name)
    have = 0
    for p in (stead_dir / name, stead_dir / (name + ".download")):
        if p.is_file():
            have = max(have, p.stat().st_size)
    needed += max(0, total - have)

free = shutil.disk_usage(stead_dir).free
print(f"Still to download: {needed / 1e9:.1f} GB | free on target disk: {free / 1e9:.1f} GB")
# Google Drive mounts report free space unreliably; Drive quota is checked at drive.google.com/settings/storage
if needed > free * 0.98 and not (IN_COLAB and USE_DRIVE):
    raise RuntimeError("Not enough disk space. Free up space, point F3S_DATA_DIR elsewhere, or set DOWNLOAD_WAVEFORMS = False.")

## 6. Download

In [ ]:
for name in files:
    download(name)

if not DOWNLOAD_WAVEFORMS and not (stead_dir / "waveforms.hdf5").is_file():
    print("Waveforms skipped. Set DOWNLOAD_WAVEFORMS = True and run again to get them.")

## 7. Verify

Checks the columns the project needs (spec section 2). Column names can differ between SeisBench versions, so missing ones are reported, not assumed.

In [ ]:
import pandas as pd

REQUIRED_COLUMNS = [
    "trace_name",
    "trace_category",
    "source_id",
    "source_magnitude",
    "source_magnitude_type",
    "trace_p_arrival_sample",
]

meta = pd.read_csv(stead_dir / "metadata.csv", low_memory=False)
print(f"Metadata rows: {len(meta):,} | columns: {meta.shape[1]}")

missing = [c for c in REQUIRED_COLUMNS if c not in meta.columns]
print("Missing required columns:", missing or "none")
if missing:
    print("All columns:", list(meta.columns))

In [ ]:
# Quick look at how much data the spec's filters leave (spec section 2)
cols = set(meta.columns)
if {"trace_category", "source_magnitude", "source_id"} <= cols:
    eq = meta[meta["trace_category"] == "earthquake_local"]
    eq = eq[eq["source_magnitude"].notna()]
    m2 = eq[eq["source_magnitude"] >= 2.0]
    print(f"earthquake_local:            {len(meta[meta['trace_category'] == 'earthquake_local']):,}")
    print(f"  with magnitude:            {len(eq):,}")
    print(f"  with M >= 2.0:             {len(m2):,} traces, {m2['source_id'].nunique():,} events")
    if "source_magnitude_type" in cols:
        print("\nMagnitude types (M >= 2.0):")
        print(m2["source_magnitude_type"].value_counts(dropna=False).head(10).to_string())

In [ ]:
if (stead_dir / "waveforms.hdf5").is_file():
    import seisbench
    import seisbench.data as sbd

    assert Path(seisbench.cache_root).resolve() == (data_dir / "seisbench").resolve(), seisbench.cache_root

    data = sbd.STEAD()  # loads from cache, no download
    print(data)
    print("Component order:", data.component_order)

    i = int(data.metadata.index[data.metadata["trace_category"] == "earthquake_local"][0])
    x = data.get_waveforms(i)
    # STEAD's metadata.csv has no sampling-rate column; SeisBench takes it from the HDF5 data_format
    print("Data format:", data.data_format)
    if "trace_sampling_rate_hz" in data.metadata.columns:
        print("Sampling rates (Hz):", data.metadata["trace_sampling_rate_hz"].unique())
    print(f"Sample trace {i}: shape {x.shape}")
else:
    print("waveforms.hdf5 not present; skipping SeisBench load check.")